##### when we were converting Documents,JSON,PDF and other stuff we were using Recurssive Charter Text slpitter and we we using chunk size , chunk overlap and other parameters 

##### but the problem with that is :
#####    langchian is a framework to build RAG applications. langchian can also be used to build Agentic AI applications . paris is the capital of france 

##### After appluying the recurssiveCharactertextsliptter we get chunks:here chunks are also seperated even though they are similar but 

##### but the chunks can be group into meaningfull units based on Similarity and context 

##### for that sematic chunking is done 

### how does sematic chunking work?

  ####  understanding : first processes is 
   1. Document segementation :- spliting the document into the sentence or paragraph
   2. apply sentece embedding :- each sentec is converted into a vector represntation 
   3. Find the cosign Similairty between adjecent sentences 
   4. if found the sentences similar to the adjecent the Merge and if they satisfy the thershould the merge them 
                                


In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
model = SentenceTransformer('all-miniLM-L6-v2')

In [6]:
text="""Langchain is a framework for building applications withLLms 
Langchain provides modular Abstractions tp combine LLMS with tools like OpenAi and Pinecone.
you can create chains,agents, memeory, and retrivers
the effiel tower is located in pune 
France is a pouplar tourist destination """

sentences=[s.strip() for s in text.split("\n") if s.strip()]

### Step 2 : Embed each sentence
embeddings = model.encode(sentences)

### Initializeing the intial parameters
threshold = 0.7

chunks=[]
current_chunk=[sentences[0]]

### Sematic grouping based on threshold
for i in range (1,len(sentences)):
    sim=cosine_similarity(
        [embeddings[i-1]],
        [embeddings[i]]    
        )[0][0]
    if sim>=threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk=[sentences[i]]
# Append the last chunk

chunks.append(" ".join(current_chunk))

# output the chunks
print("\n sematic chunks :")
for idx,chunk in enumerate(chunks):
    print(f"\nChunk{idx+1}:\n{chunk}")


 sematic chunks :

Chunk1:
Langchain is a framework for building applications withLLms Langchain provides modular Abstractions tp combine LLMS with tools like OpenAi and Pinecone.

Chunk2:
you can create chains,agents, memeory, and retrivers

Chunk3:
the effiel tower is located in pune

Chunk4:
France is a pouplar tourist destination


In [9]:
### RAG Pipeline and Modular coding 

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain.schema import Document
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from langchain.schema.runnable import RunnableLambda,RunnableMap
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [12]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [42]:
### Custom Sematic chunker with thershold 

class thresholdsematicchunker:
    def __init__ (self,model_name="all-MiniLM-L6-v2",threshold=0.7):
        self.model= SentenceTransformer(model_name)
        self.threshold = threshold

    def split(self,text:str):
        sentences=[s.strip() for s in text.split(".")if s.strip()]
        embeddings = self.model.encode(sentences)
        chunks=[]
        current_chunk=[sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i   -  1]],[embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(".".join(current_chunk)+".")
                current_chunk=[sentences[i]]
        chunks.append(".".join(current_chunk)+ ".")
        return chunks

    def split_documents(self,docs):
        result=[]
        for doc in docs :
            for chunk in self.split(doc.page_content):   
                 result.append(Document(page_content=chunk,metadata=doc.metadata))
        return result
        

In [49]:
sample_text=""""Langchain is a framework for building applications with llms.
Langchain provides modular Abstractions tp combine LLMS with tools like OpenAi and Pinecone.
you can create chains,agents, memeory, and retrivers.
the effiel tower is located in pune .
France is a pouplar tourist destination."""

In [50]:
doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='"Langchain is a framework for building applications with llms.\nLangchain provides modular Abstractions tp combine LLMS with tools like OpenAi and Pinecone.\nyou can create chains,agents, memeory, and retrivers.\nthe effiel tower is located in pune .\nFrance is a pouplar tourist destination.')

In [51]:
## chunking 
chunker = thresholdsematicchunker(threshold=0.7)
chunks=chunker.split_documents([doc])
chunks


[Document(metadata={}, page_content='"Langchain is a framework for building applications with llms.Langchain provides modular Abstractions tp combine LLMS with tools like OpenAi and Pinecone.'),
 Document(metadata={}, page_content='you can create chains,agents, memeory, and retrivers.'),
 Document(metadata={}, page_content='the effiel tower is located in pune.'),
 Document(metadata={}, page_content='France is a pouplar tourist destination.')]

In [56]:
### Vectorstore
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
embeddings=OpenAIEmbeddings()
vectorestore  = FAISS.from_documents(chunks,embeddings)
retriver = vectorestore.as_retriver()

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [60]:
## 5. Prompt templete ---
template="""answer the question based on the following context:

{context}
Question:{question}
"""
prompt =PromptTemplate.from_template(template)

In [61]:
LLM = init_chat_model(model="groq:gemma2-9b-it", temperature=0.4)

### LCEL Chain with Retrival 

rag_chain=(
    RunnableMap(
        {
        "context":lambda x:retriver.invoke(x["question"]),
        "question":lambda x:x["question"],
    }
    )
    |prompt
    |LLM
    |StrOutputParser()
)
query={"question":"what is Langchain used for?"}
result = rag_chain.invoke(query)

print(result)

NameError: name 'retriver' is not defined

In [ ]:
### Sematic_chunker_with_langchain
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain.document_loaders import TextLoader   

## Load the Documents
Loader=TextLoader("Langchain_into.txt")
docs=Loader.load()

## intialize embedding model
embeddings = OpenAIEmbeddings()

## create the sematic chunker

chunker=chunker.split_documents(docs)

## Result

for i,chunk in enumerate(chunks):
    print(f"\n chunk{i+1}:\n{chunk.page_content}")